# VERT plant-health model training

Trains the YOLO11n detector for the Raspberry Pi plant-cam and exports it as INT8 TFLite.
Generated by `edge/plant-cam/training/make_notebook.py` -- edit the scripts there, not here.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**. Then Runtime -> Run all.
On Kaggle: Settings -> Accelerator **GPU T4**, Internet **On**.

Step-by-step explanation: `edge/plant-cam/training/README.md`.

## 1. Settings

In [ ]:
RUN_NAME = "plant-v1"      # bump for each new model: plant-v2, plant-v3, ...
EPOCHS = 150               # training stops earlier on its own once it stops improving
IMGSZ = 320                # keep 320 for a Pi 4
USE_DRIVE = True           # Colab: keep everything on Google Drive so a disconnect loses nothing
UPLOAD_OWN_DATA = False    # True once you have a Label Studio "YOLO" export zip of your own photos

# Public datasets merged into training (see section 6)
USE_PLANTDOC = True        # real-field leaf photos with boxes -- the main starter set
USE_STRAWBERRY = True      # Kaggle: usmanafzaal/strawberry-disease-detection-dataset (outlines -> boxes)
USE_PLANTIFYDR = True      # Kaggle: lavaman151/plantifydr-dataset (folders -> auto boxes, train-only)
PLANTIFYDR_PER_CLASS = 400 # cap per VERT class, so lab photos don't drown out real scenes

## 2. Workspace

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")

if IN_KAGGLE:
    WORKDIR = "/kaggle/working"
elif IN_COLAB:
    WORKDIR = "/content/vert-training"
else:
    WORKDIR = os.path.abspath("vert-training")
os.makedirs(WORKDIR, exist_ok=True)
%cd {WORKDIR}

# Images stay on the fast local disk (Drive is far too slow to train from);
# only results -- runs/ and exported/ -- live on Drive, surviving a disconnect.
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    for d in ("runs", "exported"):
        target = f"/content/drive/MyDrive/vert-training/{d}"
        os.makedirs(target, exist_ok=True)
        if not os.path.islink(d):
            os.symlink(target, d)
    print("results will be saved to Google Drive: MyDrive/vert-training/")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU -- training will be very slow. Switch the runtime to a GPU."

## 3. Training scripts

Each cell below writes one file from the repo's `training/` folder.

In [ ]:
%%writefile requirements.txt
# Training machine (Linux / WSL2 / Colab with a GPU ideally), not the Pi.
# Ultralytics installs PyTorch; TensorFlow + onnx2tf for the TFLite export
# are pulled in automatically the first time export.py runs.
ultralytics>=8.3
pyyaml
pillow
minio

In [ ]:
%%writefile classes.yaml
# The single source of truth for what the model detects. Every training
# script reads this, and export.py copies it into the model's meta.json --
# which is what the Pi uses to turn detections into a health score and a
# plain-language note. Change a class here, rebuild the dataset, retrain.
#
# kind:   healthy | issue | pest
# weight: how much one detection counts against plant health (issues/pests)
# note:   what the grower reads on the dashboard when this is the top problem

classes:
  - name: healthy_leaf
    kind: healthy

  - name: chlorosis
    kind: issue
    weight: 0.6
    note: "Some leaves are yellowing -- often too much water, poor drainage, or a nitrogen shortfall."

  - name: leaf_spot_blight
    kind: issue
    weight: 1.0
    note: "Spots or blight on some leaves -- remove the worst ones and keep the foliage dry so it doesn't spread."

  - name: mildew_mold
    kind: issue
    weight: 0.8
    note: "Powdery or mouldy patches on some leaves -- more airflow and lower humidity will help."

  - name: pest
    kind: pest
    weight: 1.0
    note: "Pests or fresh pest damage spotted -- check the undersides of the leaves."

In [ ]:
%%writefile common.py
"""Small helpers shared by the training scripts."""
from __future__ import annotations

from pathlib import Path

import yaml

HERE = Path(__file__).resolve().parent
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def load_classes(path: Path = HERE / "classes.yaml") -> list[dict]:
    classes = yaml.safe_load(path.read_text(encoding="utf-8"))["classes"]
    names = [c["name"] for c in classes]
    if len(set(names)) != len(names):
        raise SystemExit(f"duplicate class names in {path}")
    return classes


def class_names() -> list[str]:
    return [c["name"] for c in load_classes()]


Box = tuple[str, float, float, float, float]


def write_source(out: Path, items: list[tuple], train_only: bool = False) -> None:
    """Writes a "source" folder in the layout build_dataset.py merges:
    images/, labels/ (YOLO txt), classes.txt. Each item is
    (image_path, boxes) or (image_path, boxes, new_stem) -- the latter when
    file names repeat across a dataset's class folders. Boxes are
    (class_name, cx, cy, w, h), normalised.

    train_only marks a source whose images must never be used to score the
    model (e.g. pre-augmented sets, where copies of one photo would land
    in both train and test and inflate the score)."""
    import shutil

    names = class_names()
    if out.exists():
        shutil.rmtree(out)
    (out / "images").mkdir(parents=True)
    (out / "labels").mkdir(parents=True)
    (out / "classes.txt").write_text("\n".join(names) + "\n", encoding="utf-8")
    if train_only:
        (out / "TRAIN_ONLY").write_text("never used for val/test -- see common.write_source\n", encoding="utf-8")
    for item in items:
        image, boxes = item[0], item[1]
        stem = item[2] if len(item) > 2 else image.stem
        shutil.copy2(image, out / "images" / f"{stem}{image.suffix.lower()}")
        lines = [f"{names.index(c)} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}" for c, cx, cy, w, h in boxes]
        (out / "labels" / f"{stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")


def safe_stem(*parts: str) -> str:
    """A file-name-safe stem built from e.g. (class folder, file stem)."""
    import re

    return "__".join(re.sub(r"[^A-Za-z0-9._-]+", "_", p).strip("_") for p in parts if p)


def print_mapping(seen: dict) -> None:
    """Prints which source class went to which VERT class -- always worth reading."""
    for target, sources in sorted(seen.items(), key=lambda kv: str(kv[0])):
        print(f"{target or 'DROPPED (unmapped)'}  ({sum(sources.values())})")
        for name, n in sources.most_common():
            print(f"    {n:6d}  {name}")

In [ ]:
%%writefile import_voc.py
"""Converts a Pascal-VOC-annotated public dataset into a "source" folder
using VERT's classes (see classes.yaml).

Two mappings are built in:

  plantdoc  PlantDoc's 30 crop-specific classes ("Tomato Early blight leaf",
            "Apple leaf", ...) folded into VERT's 5 by keyword. The summary
            printed at the end shows exactly what went where -- check it.
  pest      every object becomes `pest` (for insect datasets such as IP102's
            detection subset, whose object names are species ids)

    python import_voc.py --images PlantDoc/TRAIN --annotations PlantDoc/TRAIN \
        --mapping plantdoc --out sources/plantdoc
"""
from __future__ import annotations

import argparse
import collections
import xml.etree.ElementTree as ET
from pathlib import Path

from PIL import Image

from common import IMAGE_EXTS, class_names, write_source

# Checked in order; first match wins. Anything matching none is a plain
# leaf of some crop, i.e. healthy.
_PLANTDOC_RULES = [
    (("mite",), "pest"),
    (("mildew", "mold", "mould"), "mildew_mold"),
    (("yellow", "mosaic"), "chlorosis"),
    (("blight", "spot", "scab", "rust", "rot", "septoria"), "leaf_spot_blight"),
]


def map_plantdoc(name: str) -> str:
    lower = name.lower()
    for keywords, target in _PLANTDOC_RULES:
        if any(k in lower for k in keywords):
            return target
    return "healthy_leaf"


MAPPINGS = {"plantdoc": map_plantdoc, "pest": lambda _name: "pest"}


def find_image(images: Path, stem: str, filename: str | None) -> Path | None:
    if filename and (images / filename).exists():
        return images / filename
    for ext in IMAGE_EXTS:
        for candidate in (images / f"{stem}{ext}", images / f"{stem}{ext.upper()}"):
            if candidate.exists():
                return candidate
    return None


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--images", required=True, type=Path)
    p.add_argument("--annotations", required=True, type=Path, help="folder of VOC .xml files")
    p.add_argument("--mapping", required=True, choices=sorted(MAPPINGS))
    p.add_argument("--out", required=True, type=Path)
    args = p.parse_args()

    to_vert = MAPPINGS[args.mapping]
    valid = set(class_names())
    mapping_seen: dict[str, collections.Counter] = collections.defaultdict(collections.Counter)
    items, skipped = [], 0

    for xml_path in sorted(args.annotations.glob("*.xml")):
        root = ET.parse(xml_path).getroot()
        image = find_image(args.images, xml_path.stem, root.findtext("filename"))
        if image is None:
            skipped += 1
            continue
        # Several public datasets record width/height as 0 or wrong in the
        # XML -- the image file itself is the only trustworthy size.
        with Image.open(image) as im:
            width, height = im.size

        boxes = []
        for obj in root.iter("object"):
            source_name = (obj.findtext("name") or "").strip()
            target = to_vert(source_name)
            mapping_seen[target][source_name] += 1
            if target not in valid:
                continue
            bb = obj.find("bndbox")
            x1, y1, x2, y2 = (float(bb.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax"))
            x1, x2 = sorted((max(0.0, x1), min(float(width), x2)))
            y1, y2 = sorted((max(0.0, y1), min(float(height), y2)))
            if x2 - x1 < 2 or y2 - y1 < 2:
                continue
            boxes.append((target, (x1 + x2) / 2 / width, (y1 + y2) / 2 / height, (x2 - x1) / width, (y2 - y1) / height))
        items.append((image, boxes))

    write_source(args.out, items)
    print(f"wrote {len(items)} images to {args.out} ({skipped} annotations had no matching image)\n")
    for target, sources in sorted(mapping_seen.items()):
        print(f"{target}  ({sum(sources.values())} boxes)")
        for name, n in sources.most_common():
            print(f"    {n:6d}  {name}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile import_labelme.py
"""Converts a LabelMe-annotated dataset (one .json of polygons per image)
into a "source" folder using VERT's classes (see classes.yaml).

Built for the Strawberry Disease Detection Dataset (Afzaal et al., JBNU):

    kaggle datasets download usmanafzaal/strawberry-disease-detection-dataset -p raw/strawberry --unzip
    python import_labelme.py --root raw/strawberry --mapping strawberry --out sources/strawberry

Two things differ from VERT's own labels, and this script handles both:

  * Its outlines trace individual lesions, while VERT boxes whole leaves
    (the Pi's health score counts affected *leaves*). Lesions of the same
    class that sit close together are merged into one box, which
    approximates "one diseased leaf". --no-merge turns this off.
  * Some classes are fruit/flower diseases. They are mapped to the
    nearest VERT class below -- edit STRAWBERRY_MAP to change that; a
    class mapped to None drops every image containing it (dropping just
    the box would teach the model that the disease is background).
"""
from __future__ import annotations

import argparse
import collections
import json
from pathlib import Path

from PIL import Image

from common import IMAGE_EXTS, class_names, print_mapping, safe_stem, write_source

STRAWBERRY_MAP: dict[str, str | None] = {
    "angular_leafspot": "leaf_spot_blight",
    "leaf_spot": "leaf_spot_blight",
    "powdery_mildew_leaf": "mildew_mold",
    "powdery_mildew_fruit": "mildew_mold",
    "gray_mold": "mildew_mold",               # Botrytis -- grey fuzzy mould, mostly on fruit
    "anthracnose_fruit_rot": "leaf_spot_blight",
    "blossom_blight": "leaf_spot_blight",
}

MAPPINGS = {"strawberry": STRAWBERRY_MAP}


def normalise(label: str) -> str:
    return label.strip().lower().replace(" ", "_").replace("-", "_")


def find_image(json_path: Path, image_path: str | None) -> Path | None:
    if image_path:
        candidate = json_path.parent / Path(image_path.replace("\\", "/")).name
        if candidate.exists():
            return candidate
    for ext in IMAGE_EXTS:
        for candidate in (json_path.with_suffix(ext), json_path.with_suffix(ext.upper())):
            if candidate.exists():
                return candidate
    return None


def merge_nearby(boxes: list[list[float]], gap: float) -> list[list[float]]:
    """Unions same-class boxes whose edges come within `gap` pixels of each
    other, repeating until nothing more merges."""
    boxes = [b[:] for b in boxes]
    merged = True
    while merged:
        merged = False
        out: list[list[float]] = []
        for b in boxes:
            for o in out:
                if b[0] - gap <= o[2] and o[0] - gap <= b[2] and b[1] - gap <= o[3] and o[1] - gap <= b[3]:
                    o[0], o[1], o[2], o[3] = min(o[0], b[0]), min(o[1], b[1]), max(o[2], b[2]), max(o[3], b[3])
                    merged = True
                    break
            else:
                out.append(b)
        boxes = out
    return boxes


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--root", required=True, type=Path, help="folder searched recursively for LabelMe .json files")
    p.add_argument("--mapping", required=True, choices=sorted(MAPPINGS))
    p.add_argument("--out", required=True, type=Path)
    p.add_argument("--no-merge", action="store_true", help="keep one box per lesion instead of merging nearby ones")
    p.add_argument("--merge-gap", type=float, default=0.05, help="merge distance, as a fraction of the image's longer side")
    args = p.parse_args()

    mapping = MAPPINGS[args.mapping]
    valid = set(class_names())
    for target in mapping.values():
        if target is not None and target not in valid:
            raise SystemExit(f"mapping target {target!r} is not in classes.yaml")

    seen: dict = collections.defaultdict(collections.Counter)
    items, no_image, dropped_images, lesions, boxes_out = [], 0, 0, 0, 0
    json_files = sorted(args.root.rglob("*.json"))
    if not json_files:
        raise SystemExit(f"no .json files under {args.root} -- is this the LabelMe version of the dataset?")

    for json_path in json_files:
        try:
            doc = json.loads(json_path.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, UnicodeDecodeError):
            continue
        if "shapes" not in doc:
            continue  # not a LabelMe file (e.g. a dataset meta.json)
        image = find_image(json_path, doc.get("imagePath"))
        if image is None:
            no_image += 1
            continue
        # The image file is the only trustworthy size.
        with Image.open(image) as im:
            width, height = im.size

        per_class: dict[str, list[list[float]]] = collections.defaultdict(list)
        drop = False
        for shape in doc["shapes"]:
            source = normalise(shape.get("label", ""))
            target = mapping.get(source)
            seen[target][source] += 1
            if target is None:
                drop = True
                continue
            xs = [float(pt[0]) for pt in shape.get("points", [])]
            ys = [float(pt[1]) for pt in shape.get("points", [])]
            if len(xs) < 2:
                continue
            if shape.get("shape_type") == "circle":  # LabelMe circles are (centre, point on edge)
                r = ((xs[1] - xs[0]) ** 2 + (ys[1] - ys[0]) ** 2) ** 0.5
                xs, ys = [xs[0] - r, xs[0] + r], [ys[0] - r, ys[0] + r]
            x1, x2 = max(0.0, min(xs)), min(float(width), max(xs))
            y1, y2 = max(0.0, min(ys)), min(float(height), max(ys))
            if x2 - x1 >= 2 and y2 - y1 >= 2:
                per_class[target].append([x1, y1, x2, y2])
                lesions += 1
        if drop:
            dropped_images += 1
            continue

        boxes = []
        gap = args.merge_gap * max(width, height)
        for target, raw in per_class.items():
            for x1, y1, x2, y2 in (raw if args.no_merge else merge_nearby(raw, gap)):
                boxes.append((target, (x1 + x2) / 2 / width, (y1 + y2) / 2 / height, (x2 - x1) / width, (y2 - y1) / height))
        boxes_out += len(boxes)
        # The dataset's own split folder (train/val/test) goes in the name:
        # file names repeat across splits in some releases.
        rel = json_path.relative_to(args.root)
        items.append((image, boxes, safe_stem(*rel.parent.parts, image.stem)))

    write_source(args.out, items)
    print(f"wrote {len(items)} images to {args.out}  ({no_image} json files had no image, "
          f"{dropped_images} images dropped for unmapped classes)")
    print(f"{lesions} outlines -> {boxes_out} boxes" + ("" if args.no_merge else " after merging nearby lesions") + "\n")
    print_mapping(seen)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile import_classified.py
"""Converts a *classification* dataset -- photos sorted into one folder per
disease, no boxes -- into a "source" folder of VERT detection labels.

Built for PlantifyDr (and PlantVillage-style sets), where each photo is a
single leaf on a plain background:

    kaggle datasets download lavaman151/plantifydr-dataset -p raw/plantifydr --unzip
    python import_classified.py --root raw/plantifydr --out sources/plantifydr

What it does:
  1. A photo's class is its folder name ("Tomato___Early_blight"), mapped
     to a VERT class by keyword (RULES below). Folders matching no rule are
     skipped and listed -- read the summary and extend RULES if needed.
  2. The box is drawn automatically around the leaf, found as the
     colourful region against the plain grey/black background. When that
     fails (busy background, leaf filling the frame) the box is the whole
     photo minus a small margin -- right for a frame-filling leaf.
  3. At most --per-class photos per VERT class are kept, spread evenly
     over the source folders. These photos look nothing like a greenhouse
     camera's view, so they should *add examples* of each disease, not
     drown out the real-scene photos from PlantDoc and your own camera.
  4. The source is marked train-only: PlantifyDr ships pre-augmented
     copies of each photo, so letting it into val/test would score the
     model on near-copies of its training photos.
"""
from __future__ import annotations

import argparse
import collections
import random
from pathlib import Path

import numpy as np
from PIL import Image

from common import IMAGE_EXTS, class_names, print_mapping, safe_stem, write_source

# Checked in order against the lower-cased folder name; first match wins.
RULES: list[tuple[tuple[str, ...], str]] = [
    (("healthy",), "healthy_leaf"),
    (("mite", "insect", "pest", "aphid", "whitefly", "thrip", "miner", "worm", "beetle"), "pest"),
    (("mildew", "mold", "mould"), "mildew_mold"),
    (("yellow", "mosaic", "greening", "curl", "virus", "chlorosis", "deficien"), "chlorosis"),
    (("blight", "spot", "scab", "rust", "rot", "septoria", "canker", "measles", "esca",
      "scorch", "bacterial", "anthracnose", "blast", "melanose"), "leaf_spot_blight"),
]

# Folder names that hold splits, not classes.
_SPLIT_NAMES = {"train", "training", "valid", "validation", "val", "test", "testing", "images", "data"}


def map_folder(name: str) -> str | None:
    lower = name.lower()
    for keywords, target in RULES:
        if any(k in lower for k in keywords):
            return target
    return None


def class_folder(image: Path, root: Path) -> str:
    """The nearest parent folder that isn't a split name."""
    for parent in image.relative_to(root).parents:
        if parent.name and parent.name.lower() not in _SPLIT_NAMES:
            return parent.name
    return ""


def leaf_box(path: Path, margin: float) -> tuple[float, float, float, float]:
    """(cx, cy, w, h), normalised, around the most colourful region."""
    with Image.open(path) as im:
        im = im.convert("RGB")
        im.thumbnail((256, 256))
        hsv = np.asarray(im.convert("HSV"), dtype=np.float32)
    sat, val = hsv[:, :, 1], hsv[:, :, 2]
    # Leaves -- green, yellow or brown -- are saturated; the grey/black/white
    # backgrounds of lab-style photos (and black rotation corners) are not.
    mask = (sat > 45) & (val > 30)
    h, w = mask.shape
    rows = np.where(mask.mean(axis=1) > 0.04)[0]
    cols = np.where(mask.mean(axis=0) > 0.04)[0]
    if len(rows) and len(cols):
        y1, y2, x1, x2 = rows[0] / h, (rows[-1] + 1) / h, cols[0] / w, (cols[-1] + 1) / w
        area = (x2 - x1) * (y2 - y1)
        if 0.08 <= area <= 0.97:
            return (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1
    return 0.5, 0.5, 1 - 2 * margin, 1 - 2 * margin


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--root", required=True, type=Path)
    p.add_argument("--out", required=True, type=Path)
    p.add_argument("--per-class", type=int, default=400, help="max photos kept per VERT class")
    p.add_argument("--margin", type=float, default=0.03, help="fallback box margin when the leaf can't be found")
    p.add_argument("--seed", type=int, default=0)
    args = p.parse_args()

    valid = set(class_names())
    by_folder: dict[str, list[Path]] = collections.defaultdict(list)
    for path in args.root.rglob("*"):
        if path.suffix.lower() in IMAGE_EXTS:
            by_folder[class_folder(path, args.root)].append(path)
    if not by_folder:
        raise SystemExit(f"no images under {args.root}")

    seen: dict = collections.defaultdict(collections.Counter)
    folders_for: dict[str, list[str]] = collections.defaultdict(list)
    for folder, paths in by_folder.items():
        target = map_folder(folder)
        seen[target][folder] += len(paths)
        if target in valid:
            folders_for[target].append(folder)

    # Even spread: each VERT class's quota is shared across its folders,
    # with any folder's unused share passed on to the others.
    rng = random.Random(args.seed)
    items, fallback = [], 0
    for target, folders in sorted(folders_for.items()):
        pools = {f: rng.sample(by_folder[f], len(by_folder[f])) for f in sorted(folders)}
        chosen: list[tuple[str, Path]] = []
        while len(chosen) < args.per_class and any(pools.values()):
            for f in list(pools):
                if pools[f] and len(chosen) < args.per_class:
                    chosen.append((f, pools[f].pop()))
        for folder, path in chosen:
            try:
                box = leaf_box(path, args.margin)
            except OSError:
                continue
            if box[2] >= 1 - 2 * args.margin - 1e-6 and box[3] >= 1 - 2 * args.margin - 1e-6:
                fallback += 1
            # Full relative path in the name: the same file names repeat
            # across split folders (train/valid) and class folders.
            items.append((path, [(target, *box)], safe_stem(*path.relative_to(args.root).parent.parts, path.stem)))

    write_source(args.out, items, train_only=True)
    kept = collections.Counter(b[0] for _, boxes, _ in items for b in boxes)
    print(f"wrote {len(items)} images to {args.out} (train-only); kept per class: {dict(kept)}")
    print(f"{fallback} photos used the whole-frame fallback box\n")
    print("available photos per folder (before the --per-class cap):")
    print_mapping(seen)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile build_dataset.py
"""Merges every folder under sources/ into one YOLO dataset, remapping
class ids by *name* to classes.yaml's order (so a Label Studio export
whose classes.txt is ordered differently still lines up).

A source folder looks like:  images/  labels/  classes.txt
(what import_voc.py writes, and what Label Studio's "YOLO" export unzips to).

Validation and test images come only from your own greenhouse photos
(the --holdout source, default `own`) when you have them: a score on
PlantDoc photos says little about how the model does on *your* plants.
Public data then goes entirely to training.

    python build_dataset.py --sources sources --out datasets/vert-plant
"""
from __future__ import annotations

import argparse
import collections
import hashlib
import os
import shutil
from pathlib import Path

import yaml

from common import IMAGE_EXTS, class_names


def split_for(name: str, val_pct: int, test_pct: int) -> str:
    # Hash-based, so an image keeps its split as more data is added --
    # a test image never quietly leaks into training on the next rebuild.
    bucket = int(hashlib.sha1(name.encode()).hexdigest(), 16) % 100
    if bucket < test_pct:
        return "test"
    if bucket < test_pct + val_pct:
        return "val"
    return "train"


def link_or_copy(src: Path, dst: Path) -> None:
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--sources", type=Path, default=Path("sources"))
    p.add_argument("--out", type=Path, default=Path("datasets/vert-plant"))
    p.add_argument("--holdout", default="own", help="source folder whose images supply val/test")
    p.add_argument("--val-pct", type=int, default=20)
    p.add_argument("--test-pct", type=int, default=20)
    p.add_argument("--repeat", type=int, default=2,
                   help="copies of each holdout-source training image, so a few hundred of your own "
                        "photos aren't drowned out by thousands of public ones")
    args = p.parse_args()

    names = class_names()
    sources = sorted(d for d in args.sources.iterdir() if (d / "images").is_dir())
    if not sources:
        raise SystemExit(f"no source folders with images/ under {args.sources}")
    have_holdout = any(s.name == args.holdout for s in sources)
    if not have_holdout:
        print(f"note: no '{args.holdout}' source yet -- val/test will be drawn from all sources. "
              "Fine for a first model; replace with your own photos as soon as you have them.")

    if args.out.exists():
        shutil.rmtree(args.out)
    for split in ("train", "val", "test"):
        (args.out / "images" / split).mkdir(parents=True)
        (args.out / "labels" / split).mkdir(parents=True)

    stats = collections.defaultdict(collections.Counter)
    for src in sources:
        src_names = (src / "classes.txt").read_text(encoding="utf-8").split()
        remap = {i: names.index(n) for i, n in enumerate(src_names) if n in names}
        unknown = [n for n in src_names if n not in names]
        if unknown:
            print(f"{src.name}: dropping boxes of classes not in classes.yaml: {unknown}")
        # A TRAIN_ONLY marker (written by import_classified.py) keeps
        # pre-augmented near-duplicates out of val/test entirely.
        train_only = (src / "TRAIN_ONLY").exists()
        is_holdout = not train_only and ((src.name == args.holdout) or not have_holdout)

        for image in sorted((src / "images").iterdir()):
            if image.suffix.lower() not in IMAGE_EXTS:
                continue
            label = src / "labels" / f"{image.stem}.txt"
            if not label.exists():
                stats[src.name]["skipped_unlabelled"] += 1
                continue
            lines = []
            for line in label.read_text(encoding="utf-8").splitlines():
                parts = line.split()
                if len(parts) == 5 and int(parts[0]) in remap:
                    lines.append(" ".join([str(remap[int(parts[0])]), *parts[1:]]))
                    stats["boxes"][names[remap[int(parts[0])]]] += 1

            split = split_for(f"{src.name}/{image.name}", args.val_pct, args.test_pct) if is_holdout else "train"
            copies = args.repeat if (split == "train" and src.name == args.holdout) else 1
            for r in range(copies):
                stem = f"{src.name}__{image.stem}" + (f"__r{r}" if r else "")
                link_or_copy(image, args.out / "images" / split / f"{stem}{image.suffix.lower()}")
                (args.out / "labels" / split / f"{stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
            stats[src.name][split] += 1

    if not any(stats[s.name]["val"] for s in sources):
        raise SystemExit("no validation images -- every source is train-only. Add PlantDoc, the strawberry set, "
                         "or your own photos so there is something honest to score the model on.")

    data_yaml = {
        "path": str(args.out.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": dict(enumerate(names)),
    }
    (args.out / "data.yaml").write_text(yaml.safe_dump(data_yaml, sort_keys=False), encoding="utf-8")

    for src in sources:
        tag = " (train-only)" if (src / "TRAIN_ONLY").exists() else ""
        print(f"{src.name}{tag}: {dict(stats[src.name])}")
    print(f"boxes per class: {dict(stats['boxes'])}")
    thin = [n for n in names if stats["boxes"][n] < 150]
    if thin:
        print(f"warning: under ~150 boxes for {thin} -- expect weak results on these until you label more")
    print(f"\nwrote {args.out / 'data.yaml'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile train.py
"""Fine-tunes a YOLO nano detector on the dataset build_dataset.py wrote,
then scores it on the held-out test split.

    python train.py --data datasets/vert-plant/data.yaml --name plant-v1

320 px input is the sweet spot for a Raspberry Pi 4: roughly 4x cheaper
than 640 and still enough detail for a leaf filling a decent part of
frame. If your camera sits far from the plants and lesions are tiny,
try --imgsz 416 and check bench.py timings on the Pi.
"""
from __future__ import annotations

import argparse

from ultralytics import YOLO


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--data", required=True)
    p.add_argument("--name", required=True, help="run name, e.g. plant-v1")
    p.add_argument("--weights", default="yolo11n.pt", help="COCO-pretrained starting point (downloaded automatically)")
    p.add_argument("--imgsz", type=int, default=320)
    p.add_argument("--epochs", type=int, default=150)
    p.add_argument("--batch", type=int, default=32)
    p.add_argument("--device", default=None, help="0 for the first GPU, cpu to force CPU")
    args = p.parse_args()

    model = YOLO(args.weights)
    model.train(
        data=args.data,
        imgsz=args.imgsz,
        epochs=args.epochs,
        batch=args.batch,
        device=args.device,
        project="runs",
        name=args.name,
        patience=30,        # stop once val mAP hasn't improved for 30 epochs
        # Augmentations matched to a fixed greenhouse camera: lighting and
        # colour vary a lot over a day, geometry barely does. Keep hue
        # shifts small -- yellow vs green IS the signal for chlorosis.
        hsv_h=0.005,
        hsv_s=0.5,
        hsv_v=0.4,
        degrees=10,
        fliplr=0.5,
        flipud=0.0,
        mosaic=1.0,
        close_mosaic=15,
        seed=0,
        plots=True,
    )

    # Ultralytics appends a number if runs/<name> already exists, so ask
    # the trainer where it actually saved rather than guessing.
    best_path = model.trainer.best
    best = YOLO(str(best_path))
    metrics = best.val(data=args.data, split="test", imgsz=args.imgsz, plots=True,
                       project="runs", name=f"{args.name}-test")
    print("\n=== held-out test split ===")
    print(f"mAP50     {metrics.box.map50:.3f}")
    print(f"mAP50-95  {metrics.box.map:.3f}")
    # ap50 only has entries for classes present in the test split, in
    # ap_class_index order -- not indexed by class id.
    for ap, class_id in zip(metrics.box.ap50, metrics.box.ap_class_index):
        print(f"  {best.names[int(class_id)]:20s} AP50 {ap:.3f}")
    missing = set(best.names) - {int(c) for c in metrics.box.ap_class_index}
    if missing:
        print(f"  (no test examples of: {[best.names[i] for i in sorted(missing)]} -- label more)")
    print(f"\nconfusion matrix + PR curves: {metrics.save_dir}  -- look at them before shipping")
    print(f"next: python export.py --weights {best_path} --data {args.data} --version {args.name}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile export.py
"""Exports a trained model to INT8 TFLite and packages it with the
meta.json the Pi needs, ready for deploy/deploy-model.sh.

    python export.py --weights runs/plant-v1/weights/best.pt \
        --data datasets/vert-plant/data.yaml --version plant-v1

INT8 needs representative images to calibrate against; Ultralytics
takes them from --data. Run this on Linux, WSL2 or Colab -- the
TensorFlow toolchain behind the export is unreliable on native Windows.
"""
from __future__ import annotations

import argparse
import datetime as dt
import json
import shutil
from pathlib import Path

from ultralytics import YOLO

from common import load_classes

# Which file in Ultralytics' *_saved_model/ folder to ship, per precision.
_PRECISION_FILES = {"int8": "*_int8.tflite", "fp16": "*_float16.tflite", "fp32": "*_float32.tflite"}


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--weights", required=True, type=Path)
    p.add_argument("--data", required=True)
    p.add_argument("--version", required=True, help="e.g. plant-v1; becomes the folder name and model_version")
    p.add_argument("--imgsz", type=int, default=320)
    p.add_argument("--precision", choices=sorted(_PRECISION_FILES), default="int8")
    p.add_argument("--out", type=Path, default=Path("exported"))
    args = p.parse_args()

    model = YOLO(str(args.weights))
    classes = load_classes()
    model_names = [model.names[i] for i in sorted(model.names)]
    if model_names != [c["name"] for c in classes]:
        raise SystemExit(f"model classes {model_names} don't match classes.yaml -- rebuild the dataset and retrain")

    model.export(format="tflite", imgsz=args.imgsz, int8=args.precision == "int8",
                 half=args.precision == "fp16", data=args.data)
    saved = args.weights.parent / f"{args.weights.stem}_saved_model"
    matches = sorted(saved.glob(_PRECISION_FILES[args.precision]))
    if not matches:
        raise SystemExit(f"export produced no {_PRECISION_FILES[args.precision]} in {saved}: {[f.name for f in saved.glob('*.tflite')]}")

    version = f"{args.version}-{dt.date.today():%Y%m%d}"
    out = args.out / version
    out.mkdir(parents=True, exist_ok=True)
    shutil.copy2(matches[0], out / "model.tflite")
    meta = {
        "model_version": version,
        "trained_from": str(args.weights),
        "input_size": args.imgsz,
        "precision": args.precision,
        "created": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
        "classes": [
            {"name": c["name"], "kind": c["kind"], "weight": c.get("weight", 1.0), "note": c.get("note", "")}
            for c in classes
        ],
    }
    (out / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    size_mb = (out / "model.tflite").stat().st_size / 1e6
    print(f"\npackaged {out} ({size_mb:.1f} MB)")
    print(f"copy to the Pi:   scp -r {out} pi@<pi-ip>:~/")
    print(f"then on the Pi:   sudo /opt/plant-cam/deploy/deploy-model.sh ~/{version}")


if __name__ == "__main__":
    main()

## 4. Install

In [ ]:
!pip install -q -r requirements.txt

## 5. Your own labelled photos (optional, but this is what makes the model good)

Set `UPLOAD_OWN_DATA = True` above, then upload the zip Label Studio's **Export -> YOLO** gives you.
It's unpacked into `sources/own/`, which also becomes the test set.

In [ ]:
if UPLOAD_OWN_DATA:
    import zipfile, shutil
    if IN_COLAB:
        from google.colab import files
        uploaded = files.upload()
        zip_name = next(iter(uploaded))
    else:
        raise SystemExit("On Kaggle, add the zip as an input Dataset and set zip_name to its /kaggle/input/... path")
    shutil.rmtree("sources/own", ignore_errors=True)
    with zipfile.ZipFile(zip_name) as z:
        z.extractall("sources/own")
    print(os.listdir("sources/own"))
else:
    print("Skipping -- training on public data only.")

## 6. Public datasets

**Read the mapping summary** each import prints, and check each class went where you'd expect.

The Kaggle datasets need your API key. On Colab, open the **key icon (Secrets)** in the left sidebar
and add `KAGGLE_USERNAME` and `KAGGLE_KEY` (from kaggle.com -> Settings -> API -> Create New Token),
with notebook access switched on. On Kaggle, instead add both datasets with **Add Input** -- no key needed.

In [ ]:
import glob, shutil, subprocess

def kaggle_dataset(slug):
    """Returns a local folder holding the dataset, downloading it once if needed."""
    name = slug.split("/")[1]
    if IN_KAGGLE:
        hits = glob.glob(f"/kaggle/input/{name}") or glob.glob(f"/kaggle/input/*/{name}")
        if hits:
            return hits[0]
        raise SystemExit(f"Add the dataset {slug} with 'Add Input' in the notebook sidebar")
    target = f"/content/raw/{name}" if IN_COLAB else os.path.abspath(f"raw/{name}")
    if os.path.isdir(target) and os.listdir(target):
        print(f"{slug}: already downloaded")
        return target
    if IN_COLAB:
        from google.colab import userdata
        for secret in ("KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"):
            try:
                os.environ.setdefault(secret, userdata.get(secret))
            except Exception:
                pass  # not every secret exists -- older and newer Kaggle tokens differ
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    print(f"downloading {slug} ...")
    subprocess.run(["kaggle", "datasets", "download", slug, "-p", target, "--unzip"], check=True)
    return target

if USE_PLANTDOC:
    if not os.path.exists("PlantDoc"):
        !git clone -q https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset PlantDoc
    !python import_voc.py --images PlantDoc/TRAIN --annotations PlantDoc/TRAIN --mapping plantdoc --out sources/plantdoc
    !python import_voc.py --images PlantDoc/TEST --annotations PlantDoc/TEST --mapping plantdoc --out sources/plantdoc-test
else:
    for d in ("sources/plantdoc", "sources/plantdoc-test"):
        shutil.rmtree(d, ignore_errors=True)

In [ ]:
if USE_STRAWBERRY:
    root = kaggle_dataset("usmanafzaal/strawberry-disease-detection-dataset")
    !python import_labelme.py --root {root} --mapping strawberry --out sources/strawberry
else:
    shutil.rmtree("sources/strawberry", ignore_errors=True)

In [ ]:
if USE_PLANTIFYDR:
    root = kaggle_dataset("lavaman151/plantifydr-dataset")
    !python import_classified.py --root {root} --out sources/plantifydr --per-class {PLANTIFYDR_PER_CLASS}
else:
    shutil.rmtree("sources/plantifydr", ignore_errors=True)

Spot-check the automatic boxes on PlantifyDr photos -- each should hug the leaf (or fill the frame for close-ups).

In [ ]:
if USE_PLANTIFYDR:
    import random
    from PIL import Image as PILImage, ImageDraw
    from IPython.display import display
    sample = random.sample(sorted(glob.glob("sources/plantifydr/images/*")), 8)
    tiles = []
    for path in sample:
        im = PILImage.open(path).convert("RGB").resize((224, 224))
        label = os.path.join("sources/plantifydr/labels", os.path.splitext(os.path.basename(path))[0] + ".txt")
        draw = ImageDraw.Draw(im)
        for line in open(label):
            _, cx, cy, w, h = map(float, line.split())
            draw.rectangle([(cx - w / 2) * 224, (cy - h / 2) * 224, (cx + w / 2) * 224, (cy + h / 2) * 224], outline="red", width=3)
        tiles.append(im)
    grid = PILImage.new("RGB", (224 * 4, 224 * 2), "white")
    for i, t in enumerate(tiles):
        grid.paste(t, ((i % 4) * 224, (i // 4) * 224))
    display(grid)

## 7. Build the dataset

In [ ]:
!python build_dataset.py --sources sources --out datasets/vert-plant

## 8. Train and score on the held-out test split

Roughly 20-60 min on a T4.

In [ ]:
!python train.py --data datasets/vert-plant/data.yaml --name {RUN_NAME} --epochs {EPOCHS} --imgsz {IMGSZ} --device 0

## 9. Look at the results

The confusion matrix shows which classes get mixed up -- usually yellowing vs healthy in warm light.

In [ ]:
import glob
from IPython.display import Image, display
# Ultralytics appends a number if the run name already exists; take the newest.
best = max(glob.glob(f"runs/{RUN_NAME}*/weights/best.pt"), key=os.path.getmtime)
run_dir = os.path.dirname(os.path.dirname(best))
print("best weights:", best)
for name in ("results.png", "confusion_matrix_normalized.png", "val_batch0_pred.jpg"):
    path = os.path.join(run_dir, name)
    if os.path.exists(path):
        print(name); display(Image(filename=path, width=900))

## 10. Export INT8 TFLite and download

Produces `model.zip` holding a folder with `model.tflite` + `meta.json`. Deploy it with Step 8 of the training README.

In [ ]:
!python export.py --weights {best} --data datasets/vert-plant/data.yaml --version {RUN_NAME} --imgsz {IMGSZ}
latest = max(glob.glob(f"exported/{RUN_NAME}-*"), key=os.path.getmtime)
!cd exported && zip -qr ../model.zip {os.path.basename(latest)}
print("packaged", latest)
if IN_COLAB:
    from google.colab import files
    files.download("model.zip")
else:
    print("Kaggle: Save Version -> Save & Run All, then download model.zip from the Output tab.")